# EoH HPC Compare Demo (Multi-Seed)

Run baseline vs routed with identical settings across multiple seeds, then generate aggregate plots.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR / 'run_compare_baseline_routed.py').exists() else NB_DIR
COMPARE_ROOT = PROJECT_ROOT / 'compare_runs'

print('NOTEBOOK_DIR:', NB_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('COMPARE_ROOT:', COMPARE_ROOT)


## HPC + Experiment Settings

- Default HPC endpoint/key/model are set for ENSIA.
- `EOH_COMPARE_SEEDS` controls multi-seed runs (`comma,separated`).
- For faster debug, reduce `EOH_N_GENERATIONS` and/or use fewer seeds.

In [ ]:
# ENSIA HPC defaults
os.environ['ENSIA_VLLM_BASE'] = 'http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1'
os.environ['ENSIA_VLLM_API_KEY'] = 'my-key-ensia-2022-1030'
os.environ['ENSIA_VLLM_MODEL'] = 'QuantTrio/Qwen3-VL-235B-A22B-Instruct-AWQ'

# Meeting demo defaults
os.environ['EOH_POP_SIZE'] = '8'
os.environ['EOH_N_GENERATIONS'] = '10'
os.environ['EOH_EVAL_INSTANCES_PER_GEN'] = '256'
os.environ['EOH_HOLDOUT_INSTANCES'] = '64'
os.environ['EOH_HOLDOUT_EVAL_INTERVAL'] = '1'
os.environ['EOH_N_PROC'] = '1'
os.environ['EOH_DISABLE_NUMBA'] = '1'
os.environ['EOH_LOG_LLM_IO'] = '1'
os.environ['EOH_COMPARE_OUT'] = str(COMPARE_ROOT)
os.environ['EOH_COMPARE_SEEDS'] = '2024,2025,2026'

print({k: os.environ.get(k) for k in [
    'ENSIA_VLLM_BASE',
    'ENSIA_VLLM_API_KEY',
    'ENSIA_VLLM_MODEL',
    'EOH_POP_SIZE',
    'EOH_N_GENERATIONS',
    'EOH_EVAL_INSTANCES_PER_GEN',
    'EOH_HOLDOUT_INSTANCES',
    'EOH_HOLDOUT_EVAL_INTERVAL',
    'EOH_N_PROC',
    'EOH_DISABLE_NUMBA',
    'EOH_LOG_LLM_IO',
    'EOH_COMPARE_OUT',
    'EOH_COMPARE_SEEDS'
]})


In [ ]:
# Run baseline+routed for each seed listed in EOH_COMPARE_SEEDS
status_log = COMPARE_ROOT / 'runner_status.jsonl'
print('Runner status log:', status_log)
cmd = [sys.executable, '-u', str(PROJECT_ROOT / 'notebooks' / 'run_compare_baseline_routed.py')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
# Build aggregate plots from compare_runs (auto-discovers seed_* folders)
cmd = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'notebooks' / 'plot_run_log.py'),
    '--root',
    str(COMPARE_ROOT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
from IPython.display import Image, display

plots_dir = COMPARE_ROOT / 'plots'
plot_files = [
    'fitness_vs_gen.png',
    'operator_over_time.png',
    'invalid_and_diversity.png',
    'operator_effect_by_mode.png',
    'diagnosis_effect_routed.png',
]

print('plots_dir:', plots_dir)
for name in plot_files:
    p = plots_dir / name
    print(name, 'exists=', p.exists())
    if p.exists():
        display(Image(filename=str(p)))


In [ ]:
# Quick tail of runner status for troubleshooting
status_log = COMPARE_ROOT / 'runner_status.jsonl'
if status_log.exists():
    lines = status_log.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[-20:]))
else:
    print('No runner_status.jsonl yet')
